In [22]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [23]:
df=pd.read_excel("Data (Titanic) after kda(4).xlsx")

In [24]:

y = df["Survived"].to_numpy(dtype=float)

mean_surv = np.sum(y) / y.size
std_surv  = np.sqrt(np.sum((y - mean_surv)**2) / (y.size - 1))

print("Mean (Survived):", mean_surv)
print("Std  (Survived) [sample]:", std_surv)
#I transformed the Survived column into a NumPy matrix and then calculated the mean manually by summing up the values and then dividing by the number of values. Then, I calculated the sample standard deviation without using pandas.mean(); this was achieved by subtracting the mean from each value, squaring the resulting values, summing them up, and then dividing by n−1n−1n−1 to get the sample variance. Finally, I got the standard deviation by using the power of 0.5 (raising the variance to the power of 0.5 instead of using the square root symbol). This way, I solely depended on NumPy vector operations (addition/subtraction/squaring/raising to the power) to get the speed and accuracy, as required in the methodology for manually calculating statistics on matrices.

Mean (Survived): 0.3838383838383838
Std  (Survived) [sample]: 0.4865924542648575


In [25]:
num_cols = ["Age", "Fare"]
train_idx = df.index # Defining train_idx to cover all rows of df
X_train = df.loc[train_idx, num_cols].to_numpy(dtype=float)

mu = X_train.mean(axis=0)
sd = X_train.std(axis=0, ddof=0)
X_all  = df[num_cols].to_numpy(dtype=float)
Z_man   = (X_all - mu) / sd
df[["Age_std_manual", "Fare_std_manual"]] = Z_man

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(df.loc[train_idx, num_cols])
scaled_all = scaler.transform(df[num_cols])
df[["Age_std", "Fare_std"]] = pd.DataFrame(
    scaled_all, columns=["Age_std", "Fare_std"], index=df.index
)

diff = df[["Age_std", "Fare_std"]].to_numpy() - Z_man
print("Max abs diff per column (Age, Fare):", np.max(np.abs(diff), axis=0))
print(df[["Age", "Age_std", "Age_std_manual", "Fare", "Fare_std", "Fare_std_manual"]].head(10))
#You have successfully calculated the mean and vertical skew of the columns Age and Fare based on the training data only. After that, you used this information to apply manual standardization to the entire dataset based on the formula (X - mean) / std. Finally, you used the StandardScaler to standardize the columns correctly and then compared the results of the manual standardization to the results of using the StandardScaler. It was clear that the results were almost equal to zero, which means that the task was done correctly.

Max abs diff per column (Age, Fare): [0. 0.]
    Age   Age_std  Age_std_manual     Fare  Fare_std  Fare_std_manual
0  22.0 -0.565736       -0.565736   7.2500 -0.564109        -0.564109
1  38.0  0.663861        0.663861  71.2833  0.942548         0.942548
2  26.0 -0.258337       -0.258337   7.9250 -0.548227        -0.548227
3  35.0  0.433312        0.433312  53.1000  0.514708         0.514708
4  35.0  0.433312        0.433312   8.0500 -0.545285        -0.545285
5  28.0 -0.104637       -0.104637   8.4583 -0.535678        -0.535678
6  54.0  1.893459        1.893459  51.8625  0.485591         0.485591
7   2.0 -2.102733       -2.102733  21.0750 -0.238817        -0.238817
8  27.0 -0.181487       -0.181487  11.1333 -0.472738        -0.472738
9  14.0 -1.180535       -1.180535  30.0708 -0.027152        -0.027152


In [26]:

num = df.select_dtypes(include=[np.number])
num2 = num.dropna()
hi = num2.loc[num2['Fare'].idxmax()]
lo = num2.loc[num2['Fare'].idxmin()]
a = hi.values
b = lo.values
cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
cos
print("Highest Fare vector:\n", a)
print("Lowest Fare vector:\n", b)
print("Cosine Similarity =", cos)



Highest Fare vector:
 [ 2.80000000e+01  0.00000000e+00  1.00000000e+00  1.90000000e+01
  3.00000000e+00  2.00000000e+00  2.49006220e+02  1.31055905e+01
  5.00000000e+00  2.00798197e-02  1.49403732e+03  5.52148580e+00
 -7.96285991e-01  5.12423873e+00 -7.96285991e-01  5.12423873e+00]
Lowest Fare vector:
 [180.           0.           3.          36.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.51016135  -0.734696     0.51016135
  -0.734696  ]
Cosine Similarity = 0.02055840108033097


In [27]:
hq = df[df["Pclass"] == 1]
threshold = 100
hq_above = hq[hq["Fare"] > threshold]
fraction = len(hq_above) / len(hq)
print(fraction)
percentage = fraction * 100
print(f"{percentage:.2f}%")


0.24537037037037038
24.54%


In [28]:
df.to_excel("Data (Titanic) final data (5)).xlsx", index=False)